In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
headers = headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

In [3]:
# --------------------------------------------------
# Creating BeautifulSoup object
# --------------------------------------------------
league = 'premier-league'
tabelle = []

for n_season in range(2008,2011):
    url = f'https://www.transfermarkt.com.br/{league}/torschuetzenliste/wettbewerb/GB1/saison_id/{n_season}/altersklasse/alle/detailpos//page/1'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content,'lxml')

    # --------------------------------------------------
    # Finding the last page
    # --------------------------------------------------
    pages_info = soup.find_all('div', {'class':'pager'})
    last_page_link = pages_info[0].find_all('li',{'class':'tm-pagination__list-item tm-pagination__list-item--icon-last-page'})
    last_page_number = last_page_link[0].find('a').get('href').split('/')[-1]


    # --------------------------------------------------
    # Getting information
    # --------------------------------------------------
    for n_page in range(1,int(last_page_number)+1):
        url = f'https://www.transfermarkt.com.br/{league}/torschuetzenliste/wettbewerb/GB1/saison_id/{n_season}/altersklasse/alle/detailpos//page/{n_page}'
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content,'lxml')
        all_info = soup.find_all('table')

        player_info_1 = all_info[1].find_all('tr',{'class':'odd'})
        player_info_2 = all_info[1].find_all('tr',{'class':'even'})

        odd_even = [player_info_1,player_info_2]

        for player in odd_even:
            for row in player:
                temp = []

                # Extracting data information 
                data = row.find_all('td',{'class':'zentriert'})

                pos = int(data[0].string)
                country = data[1].find('img').get('alt')
                age = int(data[2].string)
                name = data[4].find('a').get('title')
                matches = int(data[4].find('a').string)
                goals = int(data[5].find('a').string)

                try:team = data[3].find('a').get('title')
                except AttributeError: team = data[3].string
                
                # Creating season key
                season_key = f'PL-{n_season}'

                temp.append(season_key)
                temp.append(pos)
                temp.append(country)
                temp.append(age)
                temp.append(name)
                temp.append(team)
                temp.append(matches)
                temp.append(goals)

                tabelle.append(temp)

head = (['season_id','pos','country','age','player_name','team','matches','goals'])
df_tabelle = pd.DataFrame(tabelle, columns=head)
df_tabelle.sort_values(by=['season_id','pos'],inplace=True, ignore_index=True)
display(df_tabelle)

,season_id,pos,country,age,player_name,team,matches,goals
0,PL-2008,1,França,30,Nicolas Anelka,Chelsea FC,37,19
1,PL-2008,2,Portugal,24,Cristiano Ronaldo,Manchester United FC,33,18
2,PL-2008,3,Inglaterra,28,Steven Gerrard,FC Liverpool,31,16
3,PL-2008,4,Brasil,25,Robinho,Manchester City FC,31,14
4,PL-2008,5,Espanha,25,Fernando Torres,FC Liverpool,24,14
...,...,...,...,...,...,...,...,...
805,PL-2010,271,País de Gales,20,Aaron Ramsey,FC Arsenal,7,1
806,PL-2010,272,Itália,19,Federico Macheda,Manchester United FC,7,1
807,PL-2010,273,Irlanda,24,Conor Sammon,Wigan Athletic,7,1
808,PL-2010,274,Escócia,19,Grant Hanley,Blackburn Rovers,7,1
